##Подклчение к PostgreSQL и создание базы

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
!pip install psycopg2-binary

In [ ]:
import psycopg2
import pandas as pd

# I store data in local Postgresql database. I use ngrok to connect to my database
# --- UPDATE THESE VALUES ---
DB_NAME = "DB_NAME"      # Or the specific DB name you created in pgAdmin
DB_USER = "DB_USER"         # Your username
DB_PASS = "PASSWORD" # The strong password you set for 'elena'
DB_HOST = "DB_HOST" # Copy from ngrok (e.g., 0.tcp.ngrok.io)
DB_PORT = "DB_PORT"          # Copy the 5-digit number from the end of the ngrok URL
# ---------------------------

try:
    # 1. Establish the connection
    conn = psycopg2.connect(
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    print("✅ Connection successful!")

    # 2. Simple test query
    cur = conn.cursor()
    cur.execute("SELECT version();")
    record = cur.fetchone()
    print(f"You are connected to: {record}")

    # 3. (Optional) Read a table into a Pandas DataFrame
    # df = pd.read_sql_query("SELECT * FROM your_table_name", conn)
    # print(df.head())

    cur.close()
    conn.close()
    print("🔌 Connection closed.")

except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
import psycopg2
import json


# I store data in local Postgresql database. I use ngrok to connect to my database
# --- UPDATE THESE VALUES ---
DB_NAME = "DB_NAME"      # Or the specific DB name you created in pgAdmin
DB_USER = "DB_USER"         # Your username
DB_PASS = "PASSWORD" # The strong password you set for 'elena'
DB_HOST = "DB_HOST" # Copy from ngrok (e.g., 0.tcp.ngrok.io)
DB_PORT = "DB_PORT"          # Copy the 5-digit number from the end of the ngrok URL
# ---------------------------
data = []

try:
    # 1. Establish the connection
    conn = psycopg2.connect(
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    print("✅ Connection established for data extraction.")

    # 2. Create a cursor and execute the query
    cur = conn.cursor()

    # Assuming all fields are in 'addr_texts' table based on the prompt.
    # If this causes an error, we might need to adjust the table or column names.
    query = """
    SELECT
        text,
        region,
        cityname,
        citytype,
        district,
        street,
        bldnum,
        bldlist
    FROM
        addr_texts;
    """
    cur.execute(query)
    rows = cur.fetchall()

    # Get column names from cursor description
    columns = [desc[0] for desc in cur.description]

    # 3. Transform data into the desired format
    for row in rows:
        row_dict = dict(zip(columns, row))

        raw_text = row_dict.get('text', '')

        structured_address_dict = {
            "region": row_dict.get('region'),
            "cityname": row_dict.get('cityname'),
            "citytype": row_dict.get('citytype'),
            "district": row_dict.get('district'),
            "street": row_dict.get('street'),
            "bldnum": row_dict.get('bldnum'),
            "bldlist": row_dict.get('bldlist')
        }

        # Convert structured_address_dict to a JSON string
        # ensure_ascii=False allows non-ASCII characters (e.g., Cyrillic) in the JSON string
        structured_address_json = json.dumps(structured_address_dict, ensure_ascii=False)

        data.append({
            "raw_text": raw_text,
            "structured_address": structured_address_json
        })

    # 4. Close cursor and connection
    cur.close()
    conn.close()
    print(f"✅ Data fetched and transformed successfully. Total records: {len(data)}")

    # Display first few records to verify
    if data:
        print("\nFirst 3 records of the 'data' dataset:")
        for i in range(min(3, len(data))):
            print(data[i])
    else:
        print("No data was fetched from the database.")

except psycopg2.Error as e:
    print(f"❌ Database error occurred: {e}")
    print("Please check if the 'addr_texts' table exists and contains the specified columns.")
    print("You might need to adjust the query or provide table schema information.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")


In [ ]:
from collections import defaultdict
import json # Ensure json is imported in this cell

# Check if data is already grouped
if data and 'structured_addresses' in data[0] and 'structured_address' not in data[0]:
    print("✅ Data already grouped. Skipping re-grouping.")
    # No further action needed as data is already in the desired format
elif data and 'structured_address' in data[0]:
    print("🔄 Grouping structured addresses by raw text...")
    # Group structured_addresses by raw_text
    grouped_data = defaultdict(list)
    for item in data:
        raw_text = item['raw_text']
        # Parse the JSON string back to a dictionary
        structured_address_dict = json.loads(item['structured_address'])
        grouped_data[raw_text].append(structured_address_dict)

    # Convert grouped_data back to a list of dictionaries with the desired format
    processed_data = []
    for raw_text, addresses_list in grouped_data.items():
        processed_data.append({
            "raw_text": raw_text,
            "structured_addresses": addresses_list
        })

    data = processed_data
    print(f"✅ Data grouped and processed. Total unique raw_text entries: {len(data)}")
else:
    print("⚠️ Data is empty or not in an expected format for grouping. Skipping grouping.")

# Display first few records to verify the new structure
if data:
    print("\nFirst 3 records of the 'data' dataset after grouping:")
    for i in range(min(3, len(data))):
        print(data[i])
else:
    print("No data was processed.")


The `data` variable has now been transformed into the desired format, where `structured_addresses` is a list of dictionaries. Next, the `dataset.jsonl` file and the Hugging Face dataset will be updated using this new `data` structure.

### Saving data to file `dataset.json`

Here `data` is to be saved in `dataset.json` JSON Lines, where each string is a separate json-object. It is convinient for working with extensive amount of data where further data processing is excepting.

In [ ]:
import json

output_filename = "dataset.jsonl"

try:
    with open(output_filename, 'w', encoding='utf-8') as f:
        for item in data:
            # Write each dictionary as a separate JSON line
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"✅ Data successfully saved to '{output_filename}'")
except Exception as e:
    print(f"❌ Error saving data to file: {e}")


### Function fefinition `format_prompts`

This function unites raw text and structured addresses to the one entity "input - output" or "request - respond". 

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset # Import Dataset

# 1. Tokinizer upoald for checking the lenth
max_seq_length = 1024 # Base limit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-llama-3.1-8b-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# System prompt to ensure that the model knows that output must be the list of json-objects
SYSTEM_PROMPT = (
    "You are a helpful assistant that extracts all addresses from Armenian text. "
    "Your response must be strictly a valid JSON array of objects, where each object "
    "contains only the following keys: \"country\", \"city\", \"street\", \"house\", \"apartment\". "
    "Do not include any conversational filler or explanations."
)

# 2. Function for composing texts and calculates tokens
def filter_and_format(batch):
    formatted_texts = []
    keep_indices = []

    for idx, (text, addresses) in enumerate(zip(batch['raw_text'], batch['structured_addresses'])):
        # addresses — is a list of dictionaries, now it will be a list of JSON-objects
        addresses_str = json.dumps(addresses, ensure_ascii=False)

        # Template Llama-3.1
        formatted = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\nExtract all addresses from the following text:\n{text}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n{addresses_str}<|eot_id|>"
        )

        # Calculate tokens
        token_count = len(tokenizer.tokenize(formatted))

        # If text is within the limit then save it
        if token_count <= max_seq_length:
            formatted_texts.append(formatted)
            keep_indices.append(idx)

    return {"formatted_text": formatted_texts}, keep_indices


# Convert 'data' to Hugging Face Dataset
hf_dataset = Dataset.from_list(data)

# use it for text generation
processed_dataset = hf_dataset.map(
    lambda x: filter_and_format(x)[0],
    batched=True,
    remove_columns=hf_dataset.column_names # delete old columns
)

print(f"Dataset is: {len(processed_dataset)} records")


In [ ]:
import datasets

# Split the processed_dataset into training and testing sets
# For example, 90% for training, 10% for testing
if len(processed_dataset) > 1:
    split_dataset = processed_dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split_dataset['train']
    test_dataset = split_dataset['test']
    print("✅ Dataset split into training and testing sets.")
    print(f"Train dataset size: {len(train_dataset)} records")
    print(f"Test dataset size: {len(test_dataset)} records")
else:
    print("⚠️ Not enough data to split into training and testing sets. Skipping split.")
    train_dataset = processed_dataset # Use full dataset as train if split not possible
    test_dataset = None


In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

max_seq_length = 1024
dtype = None
load_in_4bit = True

# 1. Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-llama-3.1-8b-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Add LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# Corrected formatting function:
# It must handle both a single dictionary (during Unsloth's check) and a batch (during mapping).
def formatting_prompts_func(examples):
    texts = examples["formatted_text"]
    # Ensure we return a list of strings
    return texts if isinstance(texts, list) else [texts]

def simple_formatting_func(examples):
    return examples["formatted_text"]

# 3. Training Setup
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    packing = False,
    dataset_num_proc = 2,
    formatting_func = formatting_prompts_func,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        max_seq_length = max_seq_length,
    ),
)

# 4. Start Training
trainer_stats = trainer.train()

In [ ]:
# 1.Saving of LoRA-adapters and tokenizer 
model.save_pretrained("llama3_armenian_addresses_lora")
tokenizer.save_pretrained("llama3_armenian_addresses_lora")

# 2. Optional (Merge to 16bit)
# ~16 GB, but ready for vLLM or Hugging Face
# model.save_pretrained_merged("llama3_armenian_addresses_merged", tokenizer, save_method = "merged_16bit",)

# 3. Optional: Export to  GGUF (for local launch Mac/PC via llama.cpp)
# model.save_pretrained_merged("llama3_address_q4_k_m", tokenizer, save_method = "file", quantization_method = "q4_k_m")

print("Model is saved!")

In [ ]:
#Downloading the model to PC
import shutil
from google.colab import files
import os

model_directory = "llama3_armenian_addresses_lora"
zip_filename = f"{model_directory}.zip"

# Create a zip archive of the model directory
if os.path.exists(model_directory):
    print(f"Compressing '{model_directory}' into '{zip_filename}'...")
    shutil.make_archive(model_directory, 'zip', model_directory)
    print(f"✅ Successfully created '{zip_filename}'.")

    # Download the zip file
    print(f"Downloading '{zip_filename}'...")
    try:
        files.download(zip_filename)
        print("✅ Download initiated. Check your browser's downloads.")
    except Exception as e:
        print(f"❌ Error during download: {e}")

    # Optional: Clean up the zip file after download
    # os.remove(zip_filename)
    # print(f"Cleaned up '{zip_filename}'.")
else:
    print(f"❌ Error: Model directory '{model_directory}' not found. Please ensure the model was saved correctly.")


In [ ]:
import json
from unsloth import FastLanguageModel

# 1. These options should be set like for trainig
max_seq_length = 1024
dtype = None
load_in_4bit = True

# 2. Loading previously saved model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "llama3_armenian_addresses_lora", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Native Unsloth optimization for inference

# 3. System prompt loke for training
SYSTEM_PROMPT = (
    "You are a helpful assistant that extracts addresses from Armenian text "
    "and returns them strictly in JSON format with the following fields: "
    "region, cityname, citytype, district, street, bldnum, bldlist")

# 4. Function to send a request to model
def extract_address(armenian_text):
    #Using template Llama-3.1
    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\nИзвлеки адрес из следующего текста:\n{armenian_text}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    # Tokenization
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # Respose generation
    outputs = model.generate(
        **inputs,
        max_new_tokens = 1024,
        use_cache = True,
        temperature = 0.1,    # Low temperature is required for JSON
        top_p = 0.9
    )

    # Response decoding
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # JSON validation
    try:
        address_json = json.loads(response_text)
        return address_json
    except json.JSONDecodeError:
        print("Внимание: Модель вернула невалидный JSON!")
        return response_text

# --- Testing example ---
# 
sample_text = "Ես ապրում եմ Հայաստանում, Երևան քաղաքում, Ամիրյան փողոց, շենք 4, բնակարան 12։"

extracted_data = extract_address(sample_text)
print(json.dumps(extracted_data, indent=4, ensure_ascii=False))

In [ ]:
import json

print("=== Interactive mode for address extraction ===")
print("Type text in Armenian language")
print("For exit type 'выход', 'exit' or 'q'.\n")

while True:
    user_input = input("Your text: ")

    if user_input.lower().strip() in ['выход', 'exit', 'q']:
        print("\nExit")
        break

    if not user_input.strip():
        print("Empty text. Try again\n")
        continue

    print("\n⏳ Address exctraction...")
    extracted_data = extract_address(user_input)

    print("\n✅ Result:")
    print(json.dumps(extracted_data, indent=4, ensure_ascii=False))
    print("-" * 50 + "\n")